# Tkinter Showcase – Tab‑by‑Tab Walkthrough

This notebook walks through the `TkinterShowcase` example one tab at a time. Each section explains what the tab is for, how the underlying methods work, and how they fit into the overall GUI.

You can run the full app from this notebook, or just read the explanations as a teaching/learning resource.

In [ ]:
import tkinter as tk
from tkinter import ttk, scrolledtext, messagebox
from PIL import Image, ImageTk
import threading
import time


class TkinterShowcase(ttk.Frame):
    def __init__(self, master):
        super().__init__(master)

        # ---------------------------------------------------------
        # GLOBAL FONT
        # ---------------------------------------------------------
        default_font = ("Arial", 15)
        master.option_add("*Font", default_font)

        # ---------------------------------------------------------
        # WINDOW SIZE
        # ---------------------------------------------------------
        master.title("Tkinter Showcase")
        master.geometry("1100x750")

        # ---------------------------------------------------------
        # NOTEBOOK + TAB COLORING
        # ---------------------------------------------------------
        self.notebook = ttk.Notebook(self)
        self.notebook.pack(expand=True, fill="both")

        # 12 tabs → 12 shades from white to black
        self.tab_colors = [
            "#FFFFFF", "#E6E6E6", "#CCCCCC", "#B3B3B3",
            "#999999", "#808080", "#666666", "#4D4D4D",
            "#333333", "#1A1A1A", "#0D0D0D", "#000000"
        ]
        self.tab_index = 0

        # Stronger tab styling
        self.configure_tab_style()
        self.configure_checkbox_style()

        # Update colors when switching tabs
        self.notebook.bind("<<NotebookTabChanged>>", self.update_tab_colors)

        # Build all tabs
        self.build_textbox_tab()
        self.build_image_tab()
        self.build_radiobutton_tab()
        self.build_progress_tab()
        self.build_menu_tab()
        self.build_multibutton_tab()
        self.build_listbox_tab()
        self.build_focus_tab()
        self.build_checkbox_tab()
        self.build_calculator_tab()
        self.build_button_tab()

        self.pack(expand=True, fill="both")

    # ---------------------------------------------------------
    # TAB STYLE CONFIGURATION
    # ---------------------------------------------------------
    def configure_tab_style(self):
        style = ttk.Style()

        style.configure(
            "Custom.TNotebook.Tab",
            padding=[18, 10],
            borderwidth=2,
            relief="ridge"
        )

        style.map(
            "Custom.TNotebook.Tab",
            background=[("selected", "#F0F0F0")],
            expand=[("selected", [1, 1, 1, 0])]
        )

        self.notebook.configure(style="Custom.TNotebook")

    # ---------------------------------------------------------
    # TAB COLOR HANDLING
    # ---------------------------------------------------------
    def next_tab_color(self):
        color = self.tab_colors[self.tab_index % len(self.tab_colors)]
        self.tab_index += 1
        return color

    def update_tab_colors(self, event=None):
        style = ttk.Style()

        for i in range(len(self.notebook.tabs())):
            bg = self.tab_colors[i % len(self.tab_colors)]

            # Compute brightness for contrast
            brightness = int(bg[1:3], 16) + int(bg[3:5], 16) + int(bg[5:7], 16)
            fg = "white" if brightness < 300 else "black"

            # Configure tab style
            style.configure(
                f"T{i}.TNotebook.Tab",
                background=bg,
                foreground=fg,
                padding=[18, 10],
                borderwidth=2,
                relief="ridge"
            )

    # ---------------------------------------------------------
    # TAB CREATION HELPER
    # ---------------------------------------------------------
    def make_tab(self, title):
        bg = self.next_tab_color()
        style = ttk.Style()
        index = self.tab_index - 1

        style.configure(f"T{index}.TFrame", background=bg)
        frame = ttk.Frame(self.notebook, style=f"T{index}.TFrame")
        self.notebook.add(frame, text=title)
        return frame

    # ---------------------------------------------------------
    # TEXTBOX TAB
    # ---------------------------------------------------------
    def build_textbox_tab(self):
        frame = self.make_tab("Text Box")

        text_box = scrolledtext.ScrolledText(frame, wrap="word")
        text_box.pack(expand=True, fill="both")

        long_text = (
            "Tkinter is a versatile and powerful tool for building desktop "
            "applications in Python. Its simplicity and accessibility make it "
            "a popular choice for beginners and for creating quick prototypes. "
            "Tkinter handles events using an event-driven programming model."
        )
        text_box.insert(tk.END, long_text)

    # ---------------------------------------------------------
    # IMAGE TAB
    # ---------------------------------------------------------
    def build_image_tab(self):
        frame = self.make_tab("Image")

        # open your own JPG file
        self.original_image = Image.open("Dog.jpg")
        self.photo = ImageTk.PhotoImage(self.original_image)

        self.image_label = ttk.Label(frame, image=self.photo)
        self.image_label.pack(expand=True, fill="both")

        frame.bind("<Configure>", self.resize_image)

    def resize_image(self, event):
        if event.width < 50 or event.height < 50:
            return
        resized = self.original_image.resize((event.width, event.height), Image.Resampling.LANCZOS)
        self.photo = ImageTk.PhotoImage(resized)
        self.image_label.config(image=self.photo)

    # ---------------------------------------------------------
    # RADIO BUTTON TAB
    # ---------------------------------------------------------
    def build_radiobutton_tab(self):
        frame = self.make_tab("Radio Buttons")

        self.radio_var = tk.IntVar()
        result = ttk.Label(frame, text="Selected option: None")
        result.pack(pady=10)

        def update():
            val = self.radio_var.get()
            colors = {1: "red", 2: "blue", 3: "green"}
            result.config(text=f"Selected option: Option {val}", foreground=colors[val])

        for i in range(1, 4):
            ttk.Radiobutton(frame, text=f"Option {i}", variable=self.radio_var, value=i, command=update).pack(anchor="w")

    # ---------------------------------------------------------
    # PROGRESS BAR TAB
    # ---------------------------------------------------------
    def build_progress_tab(self):
        frame = self.make_tab("Progress Bar")

        self.progress = ttk.Progressbar(frame, length=300, mode="determinate")
        self.progress.pack(pady=20)

        self.progress_label = ttk.Label(frame, text="Click Start")
        self.progress_label.pack()

        ttk.Button(frame, text="Start", command=self.start_progress).pack(pady=10)

    def start_progress(self):
        def task():
            for i in range(101):
                time.sleep(0.03)
                self.progress["value"] = i
                self.progress_label.config(text=f"Progress: {i}%")
            self.progress_label.config(text="Task Complete!")

        threading.Thread(target=task, daemon=True).start()

    # ---------------------------------------------------------
    # MENU TAB
    # ---------------------------------------------------------
    def build_menu_tab(self):
        frame = self.make_tab("Menu")

        choices = ["Option 1", "Option 2", "Option 3", "Option 4"]
        colors = {"Option 1": "red", "Option 2": "blue", "Option 3": "green", "Option 4": "purple"}

        var = tk.StringVar(value="Select an Option")
        label = ttk.Label(frame, text="")
        label.pack(pady=10)

        menu_btn = tk.Menubutton(frame, textvariable=var, relief="raised")
        menu = tk.Menu(menu_btn, tearoff=0)
        menu_btn["menu"] = menu
        menu_btn.pack(pady=10)

        def choose(opt):
            var.set(opt)
            label.config(text=f"Selected: {opt}", foreground=colors[opt])

        for opt in choices:
            menu.add_command(label=opt, command=lambda o=opt: choose(o))

    # ---------------------------------------------------------
    # MULTI-BUTTON TAB
    # ---------------------------------------------------------
    def build_multibutton_tab(self):
        frame = self.make_tab("Mouse Buttons")

        label = ttk.Label(frame, text="Click inside the window")
        label.pack(pady=10)

        canvas = tk.Canvas(frame, bg="white")
        canvas.pack(expand=True, fill="both")

        def draw(event, color):
            label.config(text=f"{color} click at ({event.x}, {event.y})")
            canvas.create_oval(event.x-10, event.y-10, event.x+10, event.y+10, fill=color)

        canvas.bind("<Button-1>", lambda e: draw(e, "red"))
        canvas.bind("<Button-2>", lambda e: draw(e, "blue"))
        canvas.bind("<Button-3>", lambda e: draw(e, "green"))

    # ---------------------------------------------------------
    # LISTBOX TAB
    # ---------------------------------------------------------
    def build_listbox_tab(self):
        frame = self.make_tab("Listbox")

        lb = tk.Listbox(frame, height=8)
        lb.pack(pady=10)

        for item in ["Item 1", "Item 2", "Item 3", "Item 4"]:
            lb.insert(tk.END, item)

        def on_select(event):
            sel = lb.curselection()
            if sel:
                messagebox.showinfo("Selected", lb.get(sel))

        lb.bind("<<ListboxSelect>>", on_select)

    # ---------------------------------------------------------
    # FOCUS TAB
    # ---------------------------------------------------------
    def build_focus_tab(self):
        frame = self.make_tab("Mouse Focus")

        label = ttk.Label(frame, text="Click inside the canvas")
        label.pack(pady=10)

        canvas = tk.Canvas(frame, bg="white")
        canvas.pack(expand=True, fill="both")

        def click(event):
            label.config(text=f"Clicked at ({event.x}, {event.y})")
            canvas.create_oval(event.x-2, event.y-2, event.y+2, event.y+2, fill="black")

        canvas.bind("<Button-1>", click)

    # ---------------------------------------------------------
    # CHECKBOX TAB
    # ---------------------------------------------------------
    def configure_checkbox_style(self):
        style = ttk.Style()
        style.configure(
            "BigCheck.TCheckbutton",
            font=("Arial", 22),  # bigger text
            padding=(10, 10)  # bigger click target
        )

    def build_checkbox_tab(self):
        frame = self.make_tab("Checkboxes")

        items = ["Red", "Green", "Blue", "Yellow"]
        colors = {"Red": "red", "Green": "green", "Blue": "blue", "Yellow": "gold"}

        vars = {}
        output = ttk.Frame(frame)
        output.pack(pady=10)

        def update():
            for w in output.winfo_children():
                w.destroy()
            for item, var in vars.items():
                if var.get():
                    ttk.Label(output, text=item, foreground=colors[item]).pack()

        for item in items:
            var = tk.BooleanVar()
            vars[item] = var
            ttk.Checkbutton(
                frame,
                text=item,
                variable=var,
                command=update,
                style="BigCheck.TCheckbutton"  # <-- bigger checkbox
            ).pack(anchor="w", pady=4)

    # ---------------------------------------------------------
    # CALCULATOR TAB (LARGER BUTTONS)
    # ---------------------------------------------------------
    def build_calculator_tab(self):
        frame = self.make_tab("Calculator")

        display_var = tk.StringVar()
        entry = ttk.Entry(frame, textvariable=display_var, font=("Arial", 26))
        entry.grid(row=0, column=0, columnspan=4, sticky="nsew")

        btn_font = ("Arial", 24)

        buttons = [
            '7','8','9','/',
            '4','5','6','*',
            '1','2','3','-',
            'C','0','=','+'
        ]

        def click(event):
            t = event.widget.cget("text")
            if t == "=":
                try:
                    display_var.set(str(eval(display_var.get())))
                except:
                    display_var.set("Error")
            elif t == "C":
                display_var.set("")
            else:
                display_var.set(display_var.get() + t)

        style = ttk.Style()
        style.configure("Calc.TButton", font=btn_font)

        r, c = 1, 0
        for b in buttons:
            btn = ttk.Button(frame, text=b, style="Calc.TButton")
            btn.grid(row=r, column=c, sticky="nsew", padx=4, pady=4)
            btn.bind("<Button-1>", click)
            c += 1
            if c > 3:
                c = 0
                r += 1

        for i in range(4):
            frame.columnconfigure(i, weight=1)
        for i in range(6):
            frame.rowconfigure(i, weight=1)

    # ---------------------------------------------------------
    # SIMPLE BUTTON TAB
    # ---------------------------------------------------------
    def build_button_tab(self):
        frame = self.make_tab("Button")

        ttk.Button(frame, text="Click Me", command=lambda: messagebox.showinfo("Clicked", "Hello!")).pack(pady=20)


# ---------------------------------------------------------
# MAIN
# ---------------------------------------------------------
if __name__ == "__main__":
    root = tk.Tk()
    TkinterShowcase(root)
    root.mainloop()


## Notebook structure by tab

Each tab in the `ttk.Notebook` is backed by one or more methods on `TkinterShowcase`. The helper methods `make_tab`, `next_tab_color`, and `update_tab_colors` handle the visual layout and color cycling, while the `build_*_tab` methods define the widgets and behavior for each page.

In the sections below, we’ll look at each tab conceptually rather than re‑defining the class, since the full class is already loaded and runnable in the previous cell.

## Text Box tab (`build_textbox_tab`)

This tab demonstrates a scrollable text area using `ScrolledText`. It’s a simple way to show multi‑paragraph content and talk about event‑driven GUIs.

The code inserts a short explanation string:

> "Tkinter is a versatile and powerful tool for building desktop applications in Python. Its simplicity and accessibility make it a popular choice for beginners and for creating quick prototypes. Tkinter handles events using an event-driven programming model."

Pedagogically, this tab is a nice place to discuss:
- **Event‑driven programming:** the GUI waits for user actions instead of running a linear script.
- **Widget configuration:** how `wrap="word"` and `expand=True, fill="both"` affect layout.
- **Stateful content:** inserting and later reading text from the widget.

## Image tab (`build_image_tab` and `resize_image`)

This tab shows how to load and resize an image with Pillow (`PIL`). The method `build_image_tab` opens `Dog.jpg`, wraps it in an `ImageTk.PhotoImage`, and displays it in a `ttk.Label`. The `resize_image` handler listens to `<Configure>` events and resizes the image whenever the frame changes size.

Key teaching points:
- **Integrating external libraries:** using Pillow alongside Tkinter.
- **Handling resize events:** reacting to layout changes instead of hard‑coding dimensions.
- **Performance considerations:** avoiding resize when width/height are too small.

## Radio Buttons tab (`build_radiobutton_tab`)

This tab uses `tk.IntVar` and three `ttk.Radiobutton` widgets to demonstrate mutually exclusive choices. The nested `update` function reads the current value and updates a label with both text and color.

Concepts to highlight:
- **Shared variable:** all radio buttons bind to the same `IntVar`.
- **Command callbacks:** `command=update` is triggered whenever a selection changes.
- **Mapping values to behavior:** using a dictionary to map `1/2/3` to colors.

## Progress Bar tab (`build_progress_tab` and `start_progress`)

Here you’ve got a classic progress bar demo. `build_progress_tab` sets up the bar and a label, and `start_progress` launches a background thread that slowly increments the bar from 0 to 100.

Teaching angles:
- **Threading in GUIs:** why long‑running work should not block the main loop.
- **Updating widgets from a loop:** changing `self.progress["value"]` and label text as the task advances.
- **User feedback:** ending with `"Task Complete!"` to signal completion.

## Menu tab (`build_menu_tab`)

This tab demonstrates a `Menubutton` with a drop‑down `Menu`. Each menu item updates a label and changes its color based on the selected option.

Useful discussion points:
- **StringVar for selection state:** `var` tracks the current choice.
- **Menus vs. buttons:** how a menu can compact multiple actions into one widget.
- **Lambda capture:** `lambda o=opt: choose(o)` to avoid the late‑binding gotcha.

## Mouse Buttons tab (`build_multibutton_tab`)

This tab is a playful way to show mouse events. Left, middle, and right clicks draw colored circles on a canvas and update a label with the click coordinates.

Concepts:
- **Different mouse buttons:** `<Button-1>`, `<Button-2>`, `<Button-3>`.
- **Event objects:** using `event.x` and `event.y` to place shapes.
- **Immediate visual feedback:** drawing shapes directly in response to clicks.

## Listbox tab (`build_listbox_tab`)

This tab shows a simple `Listbox` with four items. When the selection changes, a message box pops up showing the chosen item.

Teaching notes:
- **Listbox indexing:** `lb.curselection()` returns the selected indices.
- **Modal dialogs:** `messagebox.showinfo` as a quick way to show results.
- **Event binding:** `"<<ListboxSelect>>"` is a virtual event fired on selection changes.

## Mouse Focus tab (`build_focus_tab`)

This tab is a simpler canvas demo: each click draws a tiny dot and updates a label with the click coordinates. It’s a clean example of connecting mouse events to drawing operations.

You can use this tab to talk about:
- **Coordinate systems:** how canvas coordinates relate to the window.
- **Incremental drawing:** each click adds a new shape without clearing the canvas.
- **Feedback loops:** showing users exactly where they clicked.

## Checkboxes tab (`configure_checkbox_style` and `build_checkbox_tab`)

This tab combines custom styling with multiple checkboxes. `configure_checkbox_style` defines a `BigCheck.TCheckbutton` style with larger font and padding, and `build_checkbox_tab` uses that style to create four color options.

When a box is checked, the `update` function rebuilds a small output area showing the selected colors in their respective foreground colors.

Teaching ideas:
- **Custom styles:** using `ttk.Style` to make widgets more touch‑friendly.
- **BooleanVar:** one variable per checkbox to track on/off state.
- **Dynamic layout:** destroying and recreating labels based on current selections.

## Calculator tab (`build_calculator_tab`)

This tab is a mini calculator built with a grid of buttons and a large entry field. Each button is a `ttk.Button` with the `Calc.TButton` style, and clicks are handled by a single `click` function that inspects the button text.

Highlights:
- **Grid layout:** using `row`/`column` and `columnconfigure`/`rowconfigure` for responsive sizing.
- **Centralized event handler:** one function handles all button clicks.
- **Expression evaluation:** `eval` is used here for brevity—great chance to discuss safety and alternatives.

## Button tab (`build_button_tab`)

The final tab is intentionally simple: a single `Click Me` button that shows a `Hello!` message box. It’s a minimal example of wiring a button to a callback.

This is a good place to reinforce:
- **Command callbacks:** `command=lambda: messagebox.showinfo(...)`.
- **Small, testable actions:** how even a tiny tab can be a self‑contained demo.
- **User interaction patterns:** click → dialog → close.

## Running the full app from the notebook

The full `TkinterShowcase` class and `main` block are already defined in the code cell above. To run the GUI from this notebook:

1. Make sure `Dog.jpg` is in the same directory as the notebook.
2. Run the big code cell that defines `TkinterShowcase` and the `if __name__ == "__main__"` block.
3. The Tkinter window will open with all tabs available.

If you prefer not to run the GUI inside Jupyter, you can copy that cell into a standalone `.py` file and run it with `python TkinterShowcase.py`.